# Module 2 — Travel filter with the local LLM

**Goal:** decide whether a message is about **travel** (the chatbot handles it) or **not** (the chatbot replies politely that it only helps with travel).

```
"9adech billet l paris ?"     →  TRAVEL  →  continue the pipeline
"how do I make couscous?"     →  OTHER   →  "Sorry, I can only help with travel questions."
```

### Why an LLM here (instead of the One-Class SVM in the brief)
"Travel" is a **very broad, fuzzy topic**: a message can be about travel without looking like any of our training examples ("my flight was cancelled, what are my rights?"), and a message can mention a city without being about travel ("who won the match in Sousse?"). An LLM already understands meaning in general, so it can judge these cases without training. This was a deliberate project decision; the report states it.

### How an LLM can classify
We don't train anything. We **ask** the model, with two messages:
1. a **system message** (the instructions): what counts as travel, which languages to expect (including Tunisian Arabizi), and *"answer with exactly one word: TRAVEL or OTHER"*;
2. the **user's message**.

Settings that make it behave like a classifier:
- `temperature = 0`: the model always picks its most likely answer, so the **same message always gets the same answer** (reproducible);
- `max_tokens = 5`: it may only write a few tokens, which keeps it fast;

**If the answer is unclear** (neither word), we count the message as **travel**: refusing a real customer is the worst mistake for the chatbot, so we fail on the safe side.

The prompt and this logic live in `src/travel_filter/llm_filter.py`, and the connection to Ollama in `src/llm/ollama_client.py` (reused later for writing answers).

### Which LLM?
The project uses **one LLM for every LLM task**: `mistral:7b-instruct` (7 billion parameters, runs partly on CPU on our 4 GB GPU).

We also tried `qwen3:4b`, but the version Ollama provides is a **"thinking-only" model**: it always writes its reasoning before answering, even when asked not to. That costs several seconds per message, too slow for a one-word answer, so it was dropped.

We evaluate Mistral on **val** (never test), with special attention to **Tunisian** messages and to **speed**.

## 0. Setup

Imports, and loading **val** (`data/processed/val.jsonl`). Each row has a `text`, a `lang` and the label `is_travel` (true / false): that's the truth we will compare the LLM's answers with.

In [1]:
import sys, json
from pathlib import Path
from collections import Counter

ROOT = Path.cwd()
while not (ROOT / "CLAUDE.md").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.travel_filter.llm_filter import classify_travel, PROMPT_V1, PROMPT_V2, FEW_SHOT_V2

LANGS = ["en", "fr", "tn_latin"]
MODEL = "mistral:7b-instruct"

val = [json.loads(line) for line in open(ROOT / "data" / "processed" / "val.jsonl", encoding="utf-8")]
print("VAL:", len(val), "messages")
for lang in LANGS:
    travel = sum(r["is_travel"] for r in val if r["lang"] == lang)
    other = sum(not r["is_travel"] for r in val if r["lang"] == lang)
    print(f"  {lang:9s} travel {travel:3d} | non-travel {other:3d}")
print()
print("Prompt v1 (system message) sent to the LLM in sections 1-5:\n")
print(PROMPT_V1)

VAL: 684 messages
  en        travel 191 | non-travel  34
  fr        travel 196 | non-travel  35
  tn_latin  travel 194 | non-travel  34

Prompt v1 (system message) sent to the LLM in sections 1-5:

You are a classifier for a travel agency chatbot. Decide whether the user's message is about travel: flights, hotels and accommodation, local transport (taxis, car rental, trains, buses, louages), weather at a destination, things to do at a destination, visas and travel documents, currency exchange, trip budget, or how to pay for a trip. Messages may be written in English, French, or Tunisian Arabic in Latin letters (Arabizi, for example '9adech el billet l paris?' or 'nheb hotel f jerba'). Answer with exactly one word: TRAVEL or OTHER.


## 1. First call vs next calls: why the first one is slow

Before answering anything, Ollama must **load the model** from disk into memory (GPU + RAM). That happens on the **first** call only: afterwards the model stays loaded (we ask Ollama to keep it for 15 minutes, `keep_alive`), and each call only has to *read* the message and write one word.

So we measure two things:
- the **first call** (includes loading, a few seconds; in the real chatbot it happens once, when the app starts);
- the **next calls** (what a user actually waits for).

In [2]:
SAMPLES = ["9adech billet l paris ?", "how do I make couscous?", "Quel temps fait-il à Djerba demain ?",
           "chnowa a7sen equipe f tounes ?", "hotel pas cher a sousse svp", "explain machine learning simply"]

first = classify_travel("hello", model=MODEL, version=1)
print(f"{MODEL}: first call (includes loading the model) = {first['seconds']:.1f} s")
for text in SAMPLES:
    r = classify_travel(text, model=MODEL, version=1)
    print(f"   {r['seconds']:4.2f} s   {r['raw']!r:10s} | {text}")

mistral:7b-instruct: first call (includes loading the model) = 0.9 s


   0.55 s   'TRAVEL'   | 9adech billet l paris ?


   0.41 s   'OTHER'    | how do I make couscous?


   0.67 s   'WEATHER'  | Quel temps fait-il à Djerba demain ?


   0.63 s   'TRAVEL'   | chnowa a7sen equipe f tounes ?


   0.59 s   'TRAVEL'   | hotel pas cher a sousse svp


   0.64 s   'Machine learning is a type' | explain machine learning simply


## 2. Run the filter (prompt v1) on all of val

Each of the 684 val messages is sent to Mistral. The answers are saved to `eval/predictions/travel_filter_mistral_val_v1.jsonl`, so re-running the notebook reuses them instead of waiting again (set `RERUN = True` to force new calls, e.g. after changing the prompt).

For each message we keep: the true label, the LLM's raw answer, our decision, whether the answer was **unclear** (neither TRAVEL nor OTHER), and the time taken.

In [3]:
import time
PRED_PATH = ROOT / "eval" / "predictions" / "travel_filter_mistral_val_v1.jsonl"
PRED_PATH.parent.mkdir(parents=True, exist_ok=True)
RERUN = False

if PRED_PATH.exists() and not RERUN:
    preds = [json.loads(line) for line in open(PRED_PATH, encoding="utf-8")]
    print(f"Loaded {len(preds)} saved predictions from {PRED_PATH.relative_to(ROOT)}")
else:
    preds, start = [], time.time()
    for i, row in enumerate(val, 1):
        r = classify_travel(row["text"], model=MODEL, version=1)
        answer = r["raw"].strip().upper()
        preds.append({"id": row["id"], "lang": row["lang"], "text": row["text"], "truth": row["is_travel"],
                      "pred": r["is_travel"], "raw": r["raw"],
                      "unclear": not (answer.startswith("TRAVEL") or answer.startswith("OTHER")),
                      "seconds": r["seconds"]})
        if i % 100 == 0:
            print(f"  {i}/{len(val)} done ({time.time() - start:.0f} s)")
    with open(PRED_PATH, "w", encoding="utf-8") as f:
        for p in preds:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"Done: {len(preds)} messages in {time.time() - start:.0f} s")

Loaded 684 saved predictions from eval\predictions\travel_filter_mistral_val_v1.jsonl


## 3. Metrics

The brief's metrics for this module, with **"non-travel" as the class we want to detect**:
- **Precision (non-travel)**: of the messages the filter *rejected*, how many really were non-travel?
- **Recall (non-travel)**: of the real non-travel messages, how many did the filter catch?
- **F1**: the balance of the two.
- **False rejection rate (FRR)**: of the real **travel** messages, how many were wrongly rejected? This is the **worst** error for the chatbot (a real customer gets "I only answer travel questions"), so we want it as close to 0 as possible.

Also reported: overall accuracy, the number of **unclear** answers (counted as travel by our safe-side rule, so they must be checked separately), and the **speed** per message.

In [4]:
import statistics

def filter_metrics(rows):
    tp = sum(not p["truth"] and not p["pred"] for p in rows)   # non-travel correctly rejected
    fp = sum(p["truth"] and not p["pred"] for p in rows)       # travel wrongly rejected
    fn = sum(not p["truth"] and p["pred"] for p in rows)       # non-travel wrongly accepted
    n_travel = sum(p["truth"] for p in rows)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"n": len(rows), "accuracy": sum(p["truth"] == p["pred"] for p in rows) / len(rows),
            "precision_non_travel": precision, "recall_non_travel": recall, "f1_non_travel": f1,
            "false_rejection_rate": fp / n_travel if n_travel else 0.0,
            "false_rejections": fp, "missed_non_travel": fn,
            "unclear_answers": sum(p["unclear"] for p in rows)}

results = {"all": filter_metrics(preds)}
for lang in LANGS:
    results[lang] = filter_metrics([p for p in preds if p["lang"] == lang])

print(f"{'':9s} {'acc':>7s} {'P(non)':>7s} {'R(non)':>7s} {'F1(non)':>7s} {'FRR':>6s} {'unclear':>7s}")
for key, m in results.items():
    print(f"{key:9s} {m['accuracy']:7.1%} {m['precision_non_travel']:7.1%} {m['recall_non_travel']:7.1%} "
          f"{m['f1_non_travel']:7.1%} {m['false_rejection_rate']:6.1%} {m['unclear_answers']:7d}")

times = sorted(p["seconds"] for p in preds)
speed = {"mean_s": statistics.mean(times), "median_s": statistics.median(times), "p95_s": times[int(0.95 * len(times))]}
print(f"\nSpeed per message: mean {speed['mean_s']:.2f} s | median {speed['median_s']:.2f} s | 95% of messages under {speed['p95_s']:.2f} s")

              acc  P(non)  R(non) F1(non)    FRR unclear
all         88.7%   61.6%   67.0%   64.2%   7.4%      14
en          83.6%   47.3%   76.5%   58.4%  15.2%       9
fr          93.5%   76.3%   82.9%   79.5%   4.6%       3
tn_latin    89.0%   73.7%   41.2%   52.8%   2.6%       2

Speed per message: mean 0.73 s | median 0.73 s | 95% of messages under 0.96 s


## 4. The mistakes

- **False rejections**: real travel questions the filter refused (the errors that hurt customers).
- **Missed non-travel**: non-travel messages the filter let through (the chatbot would then try to answer them as travel).
- **Unclear answers**: what Mistral wrote when it did not answer TRAVEL or OTHER.

In [5]:
def show(title, rows):
    print(f"{title} ({len(rows)}):")
    for p in rows:
        print(f"   [{p['lang']:8s}] said {p['raw']!r:12s} | {p['text']}")
    print()

show("FALSE REJECTIONS (travel refused)", [p for p in preds if p["truth"] and not p["pred"]])
show("MISSED NON-TRAVEL (let through)", [p for p in preds if not p["truth"] and p["pred"]])
show("UNCLEAR ANSWERS", [p for p in preds if p["unclear"]])

FALSE REJECTIONS (travel refused) (43):
   [en      ] said 'OTHER'      | How many kilos of luggage can I take with Transavia
   [en      ] said 'OTHER'      | Is there room service at the hotel?
   [en      ] said 'OTHER'      | Does the hotel provide a baby cot?
   [en      ] said 'OTHER (Louage stations' | Where is the louage station in Sidi Bou Said
   [en      ] said 'OTHER'      | Is it cold in Douz tonight?
   [en      ] said 'OTHER'      | Will it rain in Brussels at the end of the month?
   [en      ] said 'OTHER (Currency exchange is' | What's 1500 euros worth in TND?
   [en      ] said 'OTHER'      | How many TND will I get for 500 CHF
   [en      ] said 'OTHER'      | What's 150 dinars worth in British pounds
   [en      ] said 'OTHER (Currency exchange is' | How many CAD will I get for 2000 DT?
   [en      ] said 'OTHER (Currency exchange)' | Should I buy SAR before going to Doha?
   [en      ] said 'OTHER'      | What's 20 TND worth in £?
   [en      ] said 'OTHER'      |

## 5. Save the v1 metrics

In [6]:
METRICS_PATH = ROOT / "eval" / "metrics" / "travel_filter_llm_v1.json"
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump({"model": MODEL, "prompt": PROMPT_V1, "set": "val", "metrics": results, "speed": speed},
              f, ensure_ascii=False, indent=2)
print("Saved", METRICS_PATH.relative_to(ROOT))

Saved eval\metrics\travel_filter_llm_v1.json


## 6. Prompt v2: a stricter definition, examples, and a forced answer format

Reading the v1 mistakes shows three problems:
1. **Travel questions refused**: Mistral did not count **currency conversion**, **weather/temperature**, **luggage rules**, **hotel services** and **payment** as "travel" (43 false rejections, 7.4%).
2. **Non-travel let through**: many **Tunisian** non-travel messages were called TRAVEL just because they were Tunisian or mentioned a city (`fama club natation behi fi bizerte?`, recipes, films…).
3. **Unclear answers**: 14 times Mistral wrote a topic word (`WEATHER`, `CURRENCY`, `CONGRATULATION`) instead of TRAVEL / OTHER.

Prompt v2 (in `src/travel_filter/llm_filter.py`) fixes them without any training:
1. a **stricter definition** that explicitly lists money conversion, weather, luggage rules, hotel services and payment as TRAVEL, and says cooking, health, school, tech, sport, pets… are OTHER **even when a city is mentioned**;
2. **16 solved examples** (*few-shot prompting*), all taken from the **train** set, in the 3 languages, including tricky Tunisian non-travel messages that mention a city. They are sent as previous turns of the conversation, so the model sees "message → answer" pairs before the real message;
3. a **forced answer format**: Ollama is given a JSON schema whose only allowed values are `TRAVEL` and `OTHER`, so the model *cannot* write anything else, and unclear answers disappear.

The cost: a longer prompt and a few JSON tokens, so each call is a bit slower.

In [7]:
print(PROMPT_V2, "\n")
print(len(FEW_SHOT_V2), "examples (from train):")
for text, label in FEW_SHOT_V2:
    print(f"   {label:6s} | {text}")

PRED_PATH_V2 = ROOT / "eval" / "predictions" / "travel_filter_mistral_val_v2.jsonl"
if PRED_PATH_V2.exists() and not RERUN:
    preds_v2 = [json.loads(line) for line in open(PRED_PATH_V2, encoding="utf-8")]
    print(f"\nLoaded {len(preds_v2)} saved v2 predictions")
else:
    preds_v2, start = [], time.time()
    for i, row in enumerate(val, 1):
        r = classify_travel(row["text"], model=MODEL, version=2)
        try:
            label = json.loads(r["raw"])["label"]
        except Exception:
            label = ""
        preds_v2.append({"id": row["id"], "lang": row["lang"], "text": row["text"], "truth": row["is_travel"],
                         "pred": r["is_travel"], "raw": r["raw"], "unclear": label not in ("TRAVEL", "OTHER"),
                         "seconds": r["seconds"]})
        if i % 100 == 0:
            print(f"  {i}/{len(val)} done ({time.time() - start:.0f} s)")
    with open(PRED_PATH_V2, "w", encoding="utf-8") as f:
        for p in preds_v2:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"\nDone: {len(preds_v2)} messages in {time.time() - start:.0f} s")

You are a strict classifier for a travel agency chatbot. Label the user's message TRAVEL or OTHER.
TRAVEL = anything a travel agency would answer: flights and airlines, luggage and baggage rules, hotels and their services, local transport (taxis, car rental, trains, buses, louages), the weather or temperature in a city, things to do at a destination, visas, passports and vaccines for a trip, currency exchange and converting amounts of money, trip costs and budgets, and how to pay (cards, cash, bank transfer).
OTHER = everything else: cooking, health, school, work, technology, sport, pets, home, music, films, writing messages, general knowledge, greetings, even when the message mentions a city.
Messages are in English, French, or Tunisian Arabic written in Latin letters (Arabizi).
Reply with one word only: TRAVEL or OTHER. Never write anything else. 

16 examples (from train):
   TRAVEL | What's the value of 100 Swiss francs in Tunisian dinars?
   OTHER  | My soup is too salty. How can 

  100/684 done (141 s)


  200/684 done (282 s)


  300/684 done (419 s)


  400/684 done (547 s)


  500/684 done (687 s)


  600/684 done (828 s)



Done: 684 messages in 945 s


### v1 vs v2
Same metrics as section 3, side by side. For the false rejection rate (FRR) and "unclear", lower is better; for everything else, higher is better.

In [8]:
results_v2 = {"all": filter_metrics(preds_v2)}
for lang in LANGS:
    results_v2[lang] = filter_metrics([p for p in preds_v2 if p["lang"] == lang])

print(f"{'':9s} | {'acc v1':>6s} {'acc v2':>6s} | {'F1(non) v1':>10s} {'v2':>6s} | {'FRR v1':>6s} {'v2':>6s} | {'unclear v1':>10s} {'v2':>4s}")
for key in results:
    a, b = results[key], results_v2[key]
    print(f"{key:9s} | {a['accuracy']:6.1%} {b['accuracy']:6.1%} | {a['f1_non_travel']:10.1%} {b['f1_non_travel']:6.1%} | "
          f"{a['false_rejection_rate']:6.1%} {b['false_rejection_rate']:6.1%} | {a['unclear_answers']:10d} {b['unclear_answers']:4d}")

times_v2 = sorted(p["seconds"] for p in preds_v2)
speed_v2 = {"mean_s": statistics.mean(times_v2), "median_s": statistics.median(times_v2),
            "p95_s": times_v2[int(0.95 * len(times_v2))]}
print(f"\nSpeed v1: mean {speed['mean_s']:.2f} s, 95% under {speed['p95_s']:.2f} s")
print(f"Speed v2: mean {speed_v2['mean_s']:.2f} s, 95% under {speed_v2['p95_s']:.2f} s")
print()
show("v2 FALSE REJECTIONS (travel refused)", [p for p in preds_v2 if p["truth"] and not p["pred"]])
show("v2 MISSED NON-TRAVEL (let through)", [p for p in preds_v2 if not p["truth"] and p["pred"]])

with open(ROOT / "eval" / "metrics" / "travel_filter_llm.json", "w", encoding="utf-8") as f:
    json.dump({"model": MODEL, "prompt_version": 2, "prompt": PROMPT_V2, "few_shot": FEW_SHOT_V2,
               "set": "val", "metrics": results_v2, "speed": speed_v2,
               "v1_for_comparison": {"metrics": results, "speed": speed}}, f, ensure_ascii=False, indent=2)
print("Saved eval/metrics/travel_filter_llm.json (final = v2)")

          | acc v1 acc v2 | F1(non) v1     v2 | FRR v1     v2 | unclear v1   v2
all       |  88.7%  93.4% |      64.2%  80.5% |   7.4%   6.0% |         14    0
en        |  83.6%  94.2% |      58.4%  84.0% |  15.2%   6.8% |          9    0
fr        |  93.5%  94.4% |      79.5%  84.3% |   4.6%   6.6% |          3    0
tn_latin  |  89.0%  91.7% |      52.8%  71.6% |   2.6%   4.6% |          2    0

Speed v1: mean 0.73 s, 95% under 0.96 s
Speed v2: mean 1.38 s, 95% under 1.72 s

v2 FALSE REJECTIONS (travel refused) (35):
   [en      ] said '{"label": "OTHER"}' | Should I buy dinars before going to Casablanca?
   [en      ] said '{"label": "OTHER"}' | Should I buy Saudi riyals before going to Dubai?
   [en      ] said '{"label": "OTHER"}' | What's 150 dinars worth in British pounds
   [en      ] said '{"label": "OTHER"}' | How many CAD will I get for 2000 DT?
   [en      ] said '{"label": "OTHER"}' | Should I buy SAR before going to Doha?
   [en      ] said '{"label": "OTHER"}' | What's 2

## 7. Conclusions (val set)

| | v1 (zero-shot) | **v2 (few-shot + forced format)** |
|---|---|---|
| Accuracy | 88.7% | **93.4%** |
| F1 for non-travel | 64.2% | **80.5%** |
| False rejection rate (travel refused) | 7.4% (43) | **6.0%** (35) |
| Unclear answers | 14 | **0** |
| Speed per message | 0.73 s | 1.38 s (95% under 1.72 s) |

**v2 is the filter we keep** (`classify_travel` uses it by default).

**What still goes wrong with v2:**
- **Travel refused (35):** mostly **currency conversion** ("How many CAD will I get for 2000 DT?", "a7sebli 150 pound bel dinars") and **payment** ("Does Apple Pay work in Berlin?", "Il faut prendre du liquide pour Mahdia"). Mistral still sees these as finance questions, not travel.
- **Non-travel let through (10):** all **Tunisian**, mostly "is there a good ___ in <city>?" (`fama club natation behi fi bizerte?`, `fama magasin chaussures sport behi fi djerba?`). A Tunisian city + "fama … behi" looks like a tourist question to the model.
- **Speed:** 1.4 s per message is a large part of the 2 s target, and the answer-writing LLM call comes on top of it.

**Possible next improvements (not done yet):** add a few currency/payment examples to the prompt; a small rule that sends any message containing a currency from our lexicon straight to TRAVEL; or keep the LLM filter only for messages the other modules cannot place.